In [1]:
"""
Convert CHELSA-TraCE21k monthly tasmax GeoTIFFs into one netCDF file
holding yearly means at 100-year time steps.
"""

import re
from pathlib import Path

import cftime
import rioxarray
import xarray as xr


# --- Settings ---------------------------------------------------------------

INPUT_DIR = Path.cwd() / 'data' / 'CHELSA-TraCE21k-centennial' / 'data'
OUTPUT_NC = Path("CHELSA_TraCE21k_tasmax_yearly.nc")
Path.cwd() / 'data' / 'CHELSA-TraCE21k-centennial' / 'output'

# Filename pattern: CHELSA_TraCE21k_tasmax_<MM>_<TTT>_V.1.0.tif
FILENAME = re.compile(r"CHELSA_TraCE21k_tasmax_(\d{2})_(-?\d+)_V\.1\.0\.tif")


# --- Helper -----------------------------------------------------------------

def index_to_year(ti: int) -> int:
    """
    CHELSA-TraCE21k-centennial: alle Zeitschritte zentennisch.
    t=20 → 1990 CE (jüngster Schritt im Datensatz), jeder Schritt = 100 Jahre.
    """
    return 1990 - (20 - ti) * 100


# --- Step 1: group files by time index --------------------------------------

# files_by_year[year] = list of 12 file paths, one per month
files_by_year = {}

for path in INPUT_DIR.glob("CHELSA_TraCE21k_tasmax_*.tif"):
    match = FILENAME.match(path.name)
    if not match:
        continue
    month = int(match.group(1))
    ti = int(match.group(2))
    year = index_to_year(ti)
    files_by_year.setdefault(year, []).append((month, path))

# sort each year's files by month
for year in files_by_year:
    files_by_year[year].sort()

print(f"Found {len(files_by_year)} time steps")


# --- Step 2: build a yearly mean for each time step -------------------------

# Chunks let xarray process the giant rasters in pieces (avoids running out of RAM)
CHUNKS = {"x": 4320, "y": 2088}

yearly_slices = []

for year in sorted(files_by_year):
    print(f"Processing year {year} ...")

    # open all 12 monthly files as lazy dask arrays
    monthly_arrays = []
    for month, path in files_by_year[year]:
        da = rioxarray.open_rasterio(path, chunks=CHUNKS).squeeze("band", drop=True)
        monthly_arrays.append(da)

    # stack along a new "month" dimension and take the mean
    stacked = xr.concat(monthly_arrays, dim="month")
    yearly_mean = stacked.mean(dim="month")

    # CHELSA stores temperatures as Kelvin * 10 -> apply scale factor
    yearly_mean = yearly_mean * 0.1

    # add a time coordinate (cftime handles BCE years that pandas can't)
    timestamp = cftime.DatetimeProlepticGregorian(year, 7, 1)
    yearly_mean = yearly_mean.expand_dims(time=[timestamp])

    yearly_slices.append(yearly_mean)


# --- Step 3: combine and save -----------------------------------------------

print("Combining all years into one dataset ...")
combined = xr.concat(yearly_slices, dim="time")
combined = combined.rename({"x": "lon", "y": "lat"})
combined.name = "tasmax"
combined.attrs = {
    "long_name": "Yearly mean of daily maximum near-surface air temperature",
    "units": "K",
    "source": "CHELSA-TraCE21k v1.0 (centennial)",
    "cell_methods": "time: mean (over 12 monthly climatologies)",
}

print(f"Writing {OUTPUT_NC} ...")
combined.to_netcdf(
    OUTPUT_NC,
    encoding={"tasmax": {"zlib": True, "complevel": 4, "dtype": "float32"}},
    unlimited_dims=["time"],
)
print("Done.")

ModuleNotFoundError: No module named 'cftime'